# CDOM Calibration Corrections

In [1]:
import os
import datetime
import numpy as np
import pandas as pd

#### Identify CG Instruments

In [2]:
from ooinet import M2M
from ooinet.utils import convert_time, ntp_seconds_to_datetime, unix_epoch_time

In [3]:
datasets = M2M.search_datasets(instrument='FLOR', English_names=True)

Searching https://ooinet.oceanobservatories.org/api/m2m/12576/sensor/inv


In [4]:
uids = []
for refdes in datasets['refdes'].unique():
    if refdes.startswith(('RS','CE')):
        pass
    else:
        # Get the deployments
        deployments = M2M.get_deployments(refdes)
        # Get the uids
        refdes_uids = deployments['uid'].unique()
        # Append the uids
        for uid in refdes_uids:
            uids.append(uid)

In [5]:
cg_uids = np.unique(uids)

#### Load the Corrections Table

In [6]:
corrections = pd.read_csv("/home/jovyan/WHOIGit/ooicgsn-data-tools/cdom_corrections/data/OOI-all-cals-aug-20250601.csv")
corrections['UID'] = corrections.apply(lambda x: '-'.join((x['Source'], x['OOI_Asset_Class'], str(x['SENSOR_SERIAL_NUMBER']).zfill(5))), axis=1)
corrections.head()

,Source,OOI_Asset_Class,SENSOR_MODEL,SENSOR_SERIAL_NUMBER,PREDEPLOYMENT_CALIB_DATE,OOI_File,SF_cdom,CF_cdom,CC_dark_counts_cdom,CC_scale_factor_cdom,CC_scale_factor_cdom_adjusted,CC_dark_counts_chlorophyll_a,CC_scale_factor_chlorophyll_a,CC_dark_counts_volume_scatter,CC_scale_factor_volume_scatter,UID
0,CGINS,FLORTK,BBFL2,830,2011-04-05,CGINS-FLORTK-00830__20110405.csv,5.62,0.297127,43.0,0.0920,0.153627,51.0,0.0122,48.0,0.000002,CGINS-FLORTK-00830
1,CGINS,FLORTK,BBFL2,830,2015-02-11,CGINS-FLORTK-00830__20150211.csv,5.62,0.221695,44.0,0.0744,0.092697,49.0,0.0121,47.0,0.000002,CGINS-FLORTK-00830
2,CGINS,FLORTK,BBFL2,830,2016-07-15,CGINS-FLORTK-00830__20160715.csv,5.62,0.286392,45.0,0.0656,0.105585,49.0,0.0121,46.0,0.000002,CGINS-FLORTK-00830
3,CGINS,FLORTK,BBFL2,830,2017-07-26,CGINS-FLORTK-00830__20170726.csv,5.62,0.286392,49.0,0.0605,0.097376,51.0,0.0119,48.0,0.000002,CGINS-FLORTK-00830
4,CGINS,FLORTK,BBFL2,830,2018-06-28,CGINS-FLORTK-00830__20180628.csv,5.62,0.286392,48.0,0.0606,0.097537,52.0,0.0123,47.0,0.000002,CGINS-FLORTK-00830


#### Update CDOM Calibration
Now, update the cdom calibration in the affected files as follows:

1. Load the updated calibration
2. Match the updated calibration with original calibration
3. Replace the CDOM calibration with the new CDOM calibration
4. Record the associated metadata change log
5. Save the new updated CDOM calibration

In [23]:
def search_file(directory, filename):
    for root, dirs, files in os.walk(directory):
        if filename in files:
            return os.path.join(root, filename)
    return None

def update_cdom(calibration, CC_scale_factor_cdom_adjusted, CF_cdom, SF_cdom):
    """Update the calibration file with new CDOM scaling factor"""
    # Find the row with the CDOM Scaling Factor
    index = calibration[calibration['name'] == 'CC_scale_factor_cdom'].index
    
    # Get the old CDOM Scaling Factor
    old_cdom = calibration.loc[index, 'value']
    old_cdom = float(old_cdom.values)

    # Check if the old_cdom and new cdom are the same
    if old_cdom == CC_scale_factor_cdom_adjusted:
        note = 'This calibration not affected by CDOM calibration correction'
        calibration.loc[index, 'notes'] = note
    else:
        # Update the CDOM Scaling Factor
        calibration.loc[index, 'value'] = CC_scale_factor_cdom_adjusted
    
        # Add a note conveying the correction
        note = f'New cdom scaling factor = old scaling factor ({old_cdom}) * scaling factor ({SF_cdom}) * correction factor ({CF_cdom}) [ppb counts^-1]'
        calibration.loc[index, 'notes'] = note

    return calibration

def get_deployment_by_uid(uid):
    """
    Query and return the deployment data from OOINet
    for a particular instrument uid
    """
    url = '/'.join((M2M.URLS['asset'],'asset','deployments',uid+'?editphase=ALL'))
    data = M2M.get_api(url)
    if len(data) == 0:
        return None
    else:
        df = pd.DataFrame(data)
        df.sort_values(by='deploymentNumber', inplace=True)
        df.reset_index(drop=True, inplace=True)
        # Replace nans in endTime with today's date
        today = unix_epoch_time(datetime.datetime.now())
        df = df.fillna({'endTime': today})
        return df


def get_deployment_data(uid, ooi_file):

    # Parse the cal_date from the cal_file name
    cal_date = ooi_file.split("__")[-1].split(".")[0]
    cal_date = unix_epoch_time(cal_date)

    # Request the deployment data
    deployments = get_deployment_by_uid(uid)
    
    # Find the applicable deployment data
    deployments = deployments[(deployments['startTime'] - cal_date) >= 0]
    deployments = deployments[(deployments['startTime'] - cal_date) == (deployments['startTime'] - cal_date).min()]
    if len(deployments) == 0:
        return None
    else:
        deployments = deployments.iloc[0].to_dict()

        # Get the subsite, node, sensor, deploymentNumber, startTime, endTime
        subsite = deployments['subsite']
        node = deployments['node']
        sensor = deployments['sensor']
        series = sensor.replace('0','')[-1]
        asset_id = deployments['sensor_uid']
        deployment_number = deployments['deploymentNumber']
        start_time = pd.to_datetime(convert_time(deployments['startTime'])).strftime('%Y-%m-%d %H:%M')
        end_time = pd.to_datetime(convert_time(deployments['endTime'])).strftime('%Y-%m-%d %H:%M')
    
        return subsite, node, sensor, series, asset_id, deployment_number, start_time, end_time


def get_vocab_data(refdes, series):

    subsite, node, sensor = refdes.split('-',2)
    
    # Now get the vocab
    vocab = M2M.get_vocab(refdes)
    vocab = vocab.iloc[0].to_dict()
    
    # Get the "Array/Platform/Node/Instrument" data
    array = vocab['tocL1']
    platform = vocab['tocL2']
    node = vocab['tocL3']
    instrument = vocab['instrument']
    instrument = f'{instrument}: {sensor[3:8]} Series {series}'

    return array, platform, node, instrument


def get_metadata_annotation(refdes,  deployment_number, start_time, end_time, gitHub_url):

    annotation = (f'MODIFIED COEFFICIENT: A metadata review found that the CDOM calibration values for the sensor ({refdes}) were incorrect for all of deployment {deployment_number} due to biased primary and secondary calibration standards which '
                  f'would result in delivery of incorrect L1 or L2 derived values during that time range. The erroneous calibration values have now been entered and verified as of XXXX. All derived L1 and L2 values generated '
                  f'prior to the annotation date for deployment {deployment_number} for the time range {start_time} to {end_time} should be re-requested in order to ensure utilization of correct calibration values. The updated values '
                  f'can be viewed at {gitHub_url}.')
    return annotation


def create_github_url(ooi_asset_class, ooi_file):
    base_url = 'https://github.com/ooi-integration/asset-management/blob/master/calibration'
    github_url = "/".join((base_url, ooi_asset_class, ooi_file))
    return github_url


def create_change_log(serial_number, uid, ooi_asset_class, ooi_file):

    # Find the applicable deployment for the calibration and get the
    # subsite, node, sensor, deploymentNumber, startTime, endTime
    deployments = get_deployment_data(uid, ooi_file)
    if deployments is None:
        return None
    else:
        subsite, node, sensor, series, asset_id, deployment_number, start_time, end_time = deployments

    # Generate the reference designator
    refdes = "-".join((subsite, node, sensor))

    # Next get the vocab info
    array, platform, node, instrument = get_vocab_data(refdes, series)

    # Get the url
    github_url = create_github_url(ooi_asset_class, ooi_file)

    # Create the annotation
    annotation = get_metadata_annotation(refdes, deployment_number, start_time, end_time, github_url)

    # Change type is fixed
    change_type = 'Calibration coefficients were modified'
   
    # Assemble the metadata change
    metadata = {
        'Array': array,
        'Platform': platform,
        'Node': node,
        'Instrument': instrument,
        'RefDes': refdes,
        'Asset ID': asset_id,
        'Serial Number': serial_number,
        'deployment': deployment_number,
        'gitHub changeDate': -9999999,
        'file': ooi_file,
        'URL': github_url,
        'changeType': change_type,
        'dateRangeStart': start_time,
        'dateRangeEnd': end_time,
        'annotation': annotation
    }
        
    return metadata

In [24]:
# Insantiate metadata change log
keys = ['Array', 'Platform', 'Node', 'Instrument', 'RefDes', 'Asset ID', 'Serial Number', 'deployment', 'gitHub changeDate', 'file', 'URL',
       'changeType', 'dateRangeStart', 'dateRangeEnd', 'annotation']
metadata_change_log = {x: [] for x in keys}

# Now update the files
for correction in corrections.itertuples():
    # Get the relevant values
    ooi_file = correction.OOI_File
    uid = correction.UID
    asset_class = correction.OOI_Asset_Class
    SF_cdom = correction.SF_cdom
    CF_cdom = correction.CF_cdom
    CC_scale_factor_cdom = correction.CC_scale_factor_cdom
    CC_scale_factor_cdom_adjusted = correction.CC_scale_factor_cdom_adjusted
    serial_number = correction.SENSOR_MODEL + "-" + str(correction.SENSOR_SERIAL_NUMBER)

    # Only change CG instruments
    if uid not in cg_uids:
        continue
    else:
        # Next, load the approapriate calibration file
        cal_file = search_file('/home/jovyan/WHOIGit/ooicgsn-asset-management/calibration/', ooi_file)
        calibration = pd.read_csv(cal_file, dtype=str)
    
        # Now update the calibration
        updated_calibration = update_cdom(calibration, CC_scale_factor_cdom_adjusted, CF_cdom, SF_cdom)
    
        # Save the correction
        updated_calibration.to_csv(cal_file, index=False)

        # Generate the metadata change log
        if CC_scale_factor_cdom_adjusted == CC_scale_factor_cdom:
            pass
        else:
            change_log = create_change_log(serial_number, uid, asset_class, ooi_file)
            if change_log is None:
                pass
            else:
                for key in change_log.keys():
                    metadata_change_log[key].append(change_log[key])

In [25]:
df = pd.DataFrame(metadata_change_log)
df

,Array,Platform,Node,Instrument,RefDes,Asset ID,Serial Number,deployment,gitHub changeDate,file,URL,changeType,dateRangeStart,dateRangeEnd,annotation
0,Coastal Pioneer NES,Upstream Inshore Profiler Mooring,Wire-Following Profiler,3-Wavelength Fluorometer: FLORT Series K,CP02PMUI-WFP01-04-FLORTK000,CGINS-FLORTK-00830,BBFL2-830,2,-9999999,CGINS-FLORTK-00830__20110405.csv,https://github.com/ooi-integration/asset-manag...,Calibration coefficients were modified,2014-04-12 16:42,2014-10-09 11:58,MODIFIED COEFFICIENT: A metadata review found ...
1,Coastal Pioneer NES,Upstream Inshore Profiler Mooring,Wire-Following Profiler,3-Wavelength Fluorometer: FLORT Series K,CP02PMUI-WFP01-04-FLORTK000,CGINS-FLORTK-00830,BBFL2-830,4,-9999999,CGINS-FLORTK-00830__20150211.csv,https://github.com/ooi-integration/asset-manag...,Calibration coefficients were modified,2015-05-02 00:09,2015-05-03 10:03,MODIFIED COEFFICIENT: A metadata review found ...
2,Coastal Pioneer NES,Upstream Inshore Profiler Mooring,Wire-Following Profiler,3-Wavelength Fluorometer: FLORT Series K,CP02PMUI-WFP01-04-FLORTK000,CGINS-FLORTK-00830,BBFL2-830,8,-9999999,CGINS-FLORTK-00830__20160715.csv,https://github.com/ooi-integration/asset-manag...,Calibration coefficients were modified,2016-10-04 13:55,2017-06-18 12:07,MODIFIED COEFFICIENT: A metadata review found ...
3,Coastal Pioneer NES,Upstream Inshore Profiler Mooring,Wire-Following Profiler,3-Wavelength Fluorometer: FLORT Series K,CP02PMUI-WFP01-04-FLORTK000,CGINS-FLORTK-00830,BBFL2-830,10,-9999999,CGINS-FLORTK-00830__20170726.csv,https://github.com/ooi-integration/asset-manag...,Calibration coefficients were modified,2017-11-09 17:03,2018-04-11 21:49,MODIFIED COEFFICIENT: A metadata review found ...
4,Coastal Pioneer NES,Upstream Inshore Profiler Mooring,Wire-Following Profiler,3-Wavelength Fluorometer: FLORT Series K,CP02PMUI-WFP01-04-FLORTK000,CGINS-FLORTK-00830,BBFL2-830,12,-9999999,CGINS-FLORTK-00830__20180628.csv,https://github.com/ooi-integration/asset-manag...,Calibration coefficients were modified,2018-11-04 16:43,2019-04-19 00:35,MODIFIED COEFFICIENT: A metadata review found ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
225,Global Station Papa,Flanking Subsurface Mooring B,Mooring Riser,3-Wavelength Fluorometer: FLORT Series D,GP03FLMB-RIS01-05-FLORTD000,CGINS-FLORTD-01435,BBFL2W-1435,6,-9999999,CGINS-FLORTD-01435__20160422.csv,https://github.com/ooi-integration/asset-manag...,Calibration coefficients were modified,2018-07-24 23:01,2019-09-27 23:55,MODIFIED COEFFICIENT: A metadata review found ...
226,Global Irminger Sea,Flanking Subsurface Mooring B,Mooring Riser,3-Wavelength Fluorometer: FLORT Series D,GI03FLMB-RIS01-05-FLORTD000,CGINS-FLORTD-01435,BBFL2W-1435,9,-9999999,CGINS-FLORTD-01435__20220202.csv,https://github.com/ooi-integration/asset-manag...,Calibration coefficients were modified,2022-07-02 13:43,2023-09-09 16:52,MODIFIED COEFFICIENT: A metadata review found ...
227,Coastal Pioneer NES,Central Surface Piercing Profiler Mooring,Surface Piercing Profiler,3-Wavelength Fluorometer: FLORT Series J,CP01CNSP-SP001-09-FLORTJ000,CGINS-FLORTJ-01205,BBFL2W-1205,1,-9999999,CGINS-FLORTJ-01205__20140610.csv,https://github.com/ooi-integration/asset-manag...,Calibration coefficients were modified,2015-05-07 19:22,2015-05-07 19:30,MODIFIED COEFFICIENT: A metadata review found ...
228,Coastal Pioneer NES,Inshore Surface Piercing Profiler Mooring,Surface Piercing Profiler,3-Wavelength Fluorometer: FLORT Series J,CP03ISSP-SP001-09-FLORTJ000,CGINS-FLORTJ-01208,BBFL2W-1208,1,-9999999,CGINS-FLORTJ-01208__20140610.csv,https://github.com/ooi-integration/asset-manag...,Calibration coefficients were modified,2014-12-14 23:30,2015-01-11 03:30,MODIFIED COEFFICIENT: A metadata review found ...


In [26]:
df.to_csv("/home/jovyan/WHOIGit/ooicgsn-data-tools/cdom_corrections/results/cgsn_metadata_change_log.csv", index=False)